In [1]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


/opt/anaconda3/envs/critical_inquiry_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/critical_inquiry_env/lib/python3.10/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [ ]:
# =========================
# 第一版 ver1
# =========================
input_file = Path("output_data/expanded_triplets_ver1_clean.json")
output_json = Path("output_data/triplets_with_embeddings_ver1.json")

output_json.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 第二版 ver2
# =========================
input_file = Path("output_data/expanded_triplets_ver2_clean.json")
output_json = Path("output_data/triplets_with_embeddings_ver2.json")

output_json.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 第三版 ver3
# =========================
input_file = Path("output_data/expanded_triplets_ver3_clean.json")
output_json = Path("output_data/triplets_with_embeddings_ver3.json")

output_json.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 第四版 ver4
# =========================
input_file = Path("output_data/expanded_triplets_ver4_clean.json")
output_json = Path("output_data/triplets_with_embeddings_ver4.json")

output_json.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 知识库
# =========================
input_file = Path("output_data/knowledge_triplets_clean.json")
output_json = Path("output_data/knowledge_with_embeddings.json")

output_json.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 2. 加载三元组数据
# =========================
with open(input_file, "r", encoding="utf-8") as f:
    triplets = json.load(f)

print(f"Loaded {len(triplets)} triplets.")

# =========================
# 3. 加载医学 BERT 模型
# =========================
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
model = SentenceTransformer(model_name)

# =========================
# 4. 三元组 → 文本
# =========================
def triplet_to_text(triple: dict) -> str:
    subject = str(triple.get("subject", "")).strip()
    predicate = str(triple.get("predicate", "")).strip()
    obj = str(triple.get("object", "")).strip()
    return f"{subject} {predicate} {obj}"

texts = [triplet_to_text(triple) for triple in triplets]

print("\nExample transformed triplets:")
for t in texts[:5]:
    print("-", t)

# =========================
# 5. 生成 embedding
# =========================
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"\nEmbedding shape: {embeddings.shape}")

# =========================
# 6. 写回 JSON
# =========================
triplets_with_embeddings = []

for triple, emb, text in tqdm(zip(triplets, embeddings, texts), total=len(triplets), desc="Saving"):
    item = {
        "source_row": triple.get("source_row"),
        "subject": triple.get("subject"),
        "predicate": triple.get("predicate"),
        "object": triple.get("object"),
        "triplet_text": text,
        "embedding": emb.tolist()
    }
    triplets_with_embeddings.append(item)

# =========================
# 7. Save JSON
# =========================
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(triplets_with_embeddings, f, ensure_ascii=False, indent=2)

print(f"\nSaved to: {output_json}")

Loaded 2833 triplets.


/opt/anaconda3/envs/critical_inquiry_env/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)
No sentence-transformers model found with name /Users/appleli/.cache/torch/sentence_transformers/microsoft_BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext. Creating a new one with MEAN pooling.



Example transformed triplets:
- lantus insulin treats type 1 diabetes
- lantus insulin treats type 2 diabetes
- lantus insulin is long-acting insulin
- lantus insulin provides steady insulin release over 24 hours
- lantus insulin helps maintain stable blood sugar levels


Batches: 100%|██████████| 89/89 [00:38<00:00,  2.30it/s]



Embedding shape: (2833, 768)


Saving: 100%|██████████| 2833/2833 [00:00<00:00, 21417.68it/s]



Saved to: output_data/triplets_with_embeddings_ver3.json
